In [3]:
import itertools
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings(
    "ignore",
    category=ConvergenceWarning,
)

In [4]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold


DATASET_NAME = "Whisper Base 3.0s"

DATASET_PATH = (
    "/Users/bhavaykhatri/Desktop/whisper_base/"
    "singBAP_dataset_whisper_whisper-base_3.0s.parquet"
)

TARGET_CLASSES = [
    "correct",
    "arched_back",
    "hunched_back",
    "sideways",
    "chest_breathing",
    "over_articulation",
    "under_articulation",
]


def decode_embedding(value):
    if isinstance(
        value,
        (bytes, bytearray, memoryview),
    ):
        return np.frombuffer(
            value,
            dtype=np.float32,
        ).copy()

    return np.asarray(
        value,
        dtype=np.float32,
    ).reshape(-1)


# Load dataset
df = pd.read_parquet(DATASET_PATH)

# Use the same samples as the baseline
df = df[
    df["experience"].isin(
        ["intermediate", "professional"]
    )
].copy()

df = df[
    df["condition"].isin(TARGET_CLASSES)
].copy()

df = df.reset_index(drop=True)


# Create features, labels, and recording groups
X = np.vstack(
    df["embedding"].apply(
        decode_embedding
    )
).astype(np.float32, copy=False)

y = (
    df["condition"]
    .astype(str)
    .to_numpy()
)

groups = (
    df["filename"]
    .astype(str)
    .to_numpy()
)


# Reproduce the same outer grouped split
outer_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

train_idx, test_idx = next(
    outer_splitter.split(
        X,
        y,
        groups=groups,
    )
)

X_train = X[train_idx]
X_test = X[test_idx]

y_train = y[train_idx]
y_test = y[test_idx]

train_groups = groups[train_idx]
test_groups = groups[test_idx]


print("Full dataset:", X.shape)
print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print(
    "Shared outer recordings:",
    len(
        set(train_groups)
        & set(test_groups)
    ),
)

Full dataset: (3438, 512)
Training set: (2750, 512)
Test set: (688, 512)
Shared outer recordings: 0


In [5]:
outer_train_groups = groups[train_idx]

inner_splitter = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=43,
)

feature_train_relative_idx, validation_relative_idx = next(
    inner_splitter.split(
        X_train,
        y_train,
        groups=outer_train_groups,
    )
)

X_feature_train = X_train[
    feature_train_relative_idx
]

y_feature_train = y_train[
    feature_train_relative_idx
]

X_validation = X_train[
    validation_relative_idx
]

y_validation = y_train[
    validation_relative_idx
]

feature_train_groups = outer_train_groups[
    feature_train_relative_idx
]

validation_groups = outer_train_groups[
    validation_relative_idx
]

shared_inner_recordings = (
    set(feature_train_groups)
    & set(validation_groups)
)

print(
    "Feature-selection training:",
    X_feature_train.shape,
)

print(
    "Validation:",
    X_validation.shape,
)

print(
    "Shared inner recordings:",
    len(shared_inner_recordings),
)

Feature-selection training: (2062, 512)
Validation: (688, 512)
Shared inner recordings: 0


In [6]:
SEARCH_MODEL = make_pipeline(
    StandardScaler(),
    LinearSVC(
        C=1.0,
        class_weight="balanced",
        dual=False,
        tol=1e-3,
        max_iter=20000,
        random_state=42,
    ),
)

score_cache = {}


def evaluate_feature_indices(feature_indices):
    feature_indices = np.asarray(
        feature_indices,
        dtype=int,
    )

    if feature_indices.size == 0:
        return np.nan

    cache_key = tuple(
        sorted(feature_indices.tolist())
    )

    if cache_key in score_cache:
        return score_cache[cache_key]

    model = clone(SEARCH_MODEL)

    model.fit(
        X_feature_train[:, feature_indices],
        y_feature_train,
    )

    validation_predictions = model.predict(
        X_validation[:, feature_indices]
    )

    score = f1_score(
        y_validation,
        validation_predictions,
        average="macro",
        zero_division=0,
    )

    score_cache[cache_key] = score

    return score

In [7]:
selection_results = []
selected_feature_sets = {}


def record_selection(
    method,
    family,
    feature_indices,
    validation_macro_f1,
    elapsed_time,
):
    feature_indices = np.asarray(
        feature_indices,
        dtype=int,
    )

    selected_feature_sets[method] = (
        feature_indices.copy()
    )

    selection_results.append({
        "Method": method,
        "Family": family,
        "Selected Features": len(
            feature_indices
        ),
        "Validation Macro F1": (
            validation_macro_f1
        ),
        "Selection Time (s)": (
            elapsed_time
        ),
    })

    print(
        f"{method}: "
        f"{len(feature_indices)} features, "
        f"validation Macro F1="
        f"{validation_macro_f1:.4f}, "
        f"time={elapsed_time:.2f}s"
    )

In [8]:
all_feature_indices = np.arange(
    X_feature_train.shape[1]
)

start_time = time.time()

all_features_validation_f1 = (
    evaluate_feature_indices(
        all_feature_indices
    )
)

elapsed_time = time.time() - start_time

record_selection(
    method="All Features",
    family="Baseline",
    feature_indices=all_feature_indices,
    validation_macro_f1=(
        all_features_validation_f1
    ),
    elapsed_time=elapsed_time,
)

All Features: 512 features, validation Macro F1=0.3794, time=2.69s


In [9]:
ANOVA_K_VALUES = [
    64,
    128,
    192,
    256,
    320,
    384,
    416,
    448,
    480,
    496,
]

for k in ANOVA_K_VALUES:
    if k >= X_feature_train.shape[1]:
        continue

    start_time = time.time()

    selector = SelectKBest(
        score_func=f_classif,
        k=k,
    )

    selector.fit(
        X_feature_train,
        y_feature_train,
    )

    selected_indices = (
        selector.get_support(
            indices=True
        )
    )

    validation_f1 = (
        evaluate_feature_indices(
            selected_indices
        )
    )

    elapsed_time = (
        time.time() - start_time
    )

    record_selection(
        method=f"ANOVA K={k}",
        family="ANOVA",
        feature_indices=selected_indices,
        validation_macro_f1=validation_f1,
        elapsed_time=elapsed_time,
    )

ANOVA K=64: 64 features, validation Macro F1=0.2490, time=0.12s
ANOVA K=128: 128 features, validation Macro F1=0.2845, time=0.31s
ANOVA K=192: 192 features, validation Macro F1=0.3401, time=0.56s
ANOVA K=256: 256 features, validation Macro F1=0.3467, time=0.89s
ANOVA K=320: 320 features, validation Macro F1=0.3585, time=1.25s
ANOVA K=384: 384 features, validation Macro F1=0.3591, time=1.64s
ANOVA K=416: 416 features, validation Macro F1=0.3795, time=1.97s
ANOVA K=448: 448 features, validation Macro F1=0.3595, time=2.27s
ANOVA K=480: 480 features, validation Macro F1=0.3748, time=2.51s
ANOVA K=496: 496 features, validation Macro F1=0.3728, time=2.61s


In [10]:
L1_C_VALUES = [
    0.001,
    0.003,
    0.01,
    0.03,
    0.1,
]

l1_scaler = StandardScaler()

X_feature_train_l1 = (
    l1_scaler.fit_transform(
        X_feature_train
    )
)

for c_value in L1_C_VALUES:
    start_time = time.time()

    l1_model = LinearSVC(
        C=c_value,
        penalty="l1",
        dual=False,
        class_weight="balanced",
        tol=1e-3,
        max_iter=20000,
        random_state=42,
    )

    l1_model.fit(
        X_feature_train_l1,
        y_feature_train,
    )

    selected_mask = np.any(
        np.abs(l1_model.coef_) > 1e-8,
        axis=0,
    )

    selected_indices = np.flatnonzero(
        selected_mask
    )

    if len(selected_indices) == 0:
        print(
            f"L1-SVM C={c_value} "
            "selected zero features."
        )
        continue

    validation_f1 = (
        evaluate_feature_indices(
            selected_indices
        )
    )

    elapsed_time = (
        time.time() - start_time
    )

    record_selection(
        method=f"L1-SVM C={c_value}",
        family="L1-SVM",
        feature_indices=selected_indices,
        validation_macro_f1=validation_f1,
        elapsed_time=elapsed_time,
    )

L1-SVM C=0.001 selected zero features.
L1-SVM C=0.003: 6 features, validation Macro F1=0.1746, time=0.10s
L1-SVM C=0.01: 125 features, validation Macro F1=0.2980, time=0.32s
L1-SVM C=0.03: 296 features, validation Macro F1=0.3331, time=1.42s
L1-SVM C=0.1: 464 features, validation Macro F1=0.3829, time=5.45s


In [11]:
def sequential_forward_selection(
    candidate_indices,
    maximum_selected=15,
):
    candidate_indices = list(
        map(int, candidate_indices)
    )

    selected = []
    remaining = candidate_indices.copy()

    best_overall_score = -np.inf
    best_overall_subset = None

    history = []

    maximum_steps = min(
        maximum_selected,
        len(candidate_indices),
    )

    for step in range(maximum_steps):
        best_step_feature = None
        best_step_score = -np.inf

        for feature_index in remaining:
            trial_subset = (
                selected
                + [feature_index]
            )

            score = evaluate_feature_indices(
                trial_subset
            )

            if score > best_step_score:
                best_step_score = score
                best_step_feature = (
                    feature_index
                )

        selected.append(
            best_step_feature
        )

        remaining.remove(
            best_step_feature
        )

        history.append({
            "Step": step + 1,
            "Added Feature": (
                best_step_feature
            ),
            "Selected Features": len(
                selected
            ),
            "Validation Macro F1": (
                best_step_score
            ),
        })

        print(
            f"Step {step + 1}: "
            f"added feature "
            f"{best_step_feature}, "
            f"Macro F1="
            f"{best_step_score:.4f}"
        )

        if best_step_score > best_overall_score:
            best_overall_score = (
                best_step_score
            )

            best_overall_subset = (
                selected.copy()
            )

    return (
        np.asarray(
            best_overall_subset,
            dtype=int,
        ),
        best_overall_score,
        pd.DataFrame(history),
    )

In [12]:
SFS_PREFILTER_K = 30
SFS_MAXIMUM_SELECTED = 15

sfs_prefilter = SelectKBest(
    score_func=f_classif,
    k=SFS_PREFILTER_K,
)

sfs_prefilter.fit(
    X_feature_train,
    y_feature_train,
)

sfs_candidate_indices = (
    sfs_prefilter.get_support(
        indices=True
    )
)

start_time = time.time()

(
    sequential_indices,
    sequential_validation_f1,
    sequential_history_df,
) = sequential_forward_selection(
    candidate_indices=(
        sfs_candidate_indices
    ),
    maximum_selected=(
        SFS_MAXIMUM_SELECTED
    ),
)

elapsed_time = time.time() - start_time

record_selection(
    method="Sequential Forward",
    family="Sequential",
    feature_indices=sequential_indices,
    validation_macro_f1=(
        sequential_validation_f1
    ),
    elapsed_time=elapsed_time,
)

sequential_history_df

Step 1: added feature 337, Macro F1=0.1224
Step 2: added feature 314, Macro F1=0.1775
Step 3: added feature 129, Macro F1=0.2074
Step 4: added feature 400, Macro F1=0.2129
Step 5: added feature 211, Macro F1=0.2233
Step 6: added feature 352, Macro F1=0.2152
Step 7: added feature 436, Macro F1=0.2185
Step 8: added feature 167, Macro F1=0.2203
Step 9: added feature 399, Macro F1=0.2242
Step 10: added feature 28, Macro F1=0.2345
Step 11: added feature 62, Macro F1=0.2346
Step 12: added feature 263, Macro F1=0.2303
Step 13: added feature 198, Macro F1=0.2311
Step 14: added feature 15, Macro F1=0.2296
Step 15: added feature 85, Macro F1=0.2378
Sequential Forward: 15 features, validation Macro F1=0.2378, time=2.16s


,Step,Added Feature,Selected Features,Validation Macro F1
0,1,337,1,0.122370
1,2,314,2,0.177526
2,3,129,3,0.207432
3,4,400,4,0.212945
4,5,211,5,0.223335
5,6,352,6,0.215203
6,7,436,7,0.218525
7,8,167,8,0.220309
8,9,399,9,0.224205
9,10,28,10,0.234533


In [13]:
def exhaustive_feature_selection(
    candidate_indices,
    minimum_subset_size=3,
    maximum_subset_size=5,
):
    candidate_indices = list(
        map(int, candidate_indices)
    )

    best_score = -np.inf
    best_subset = None
    evaluated_subsets = 0

    history = []

    for subset_size in range(
        minimum_subset_size,
        maximum_subset_size + 1,
    ):
        print(
            f"Testing all subsets of "
            f"size {subset_size}..."
        )

        for subset in itertools.combinations(
            candidate_indices,
            subset_size,
        ):
            score = evaluate_feature_indices(
                subset
            )

            evaluated_subsets += 1

            if score > best_score:
                best_score = score
                best_subset = subset

                history.append({
                    "Evaluated Subsets": (
                        evaluated_subsets
                    ),
                    "Subset Size": (
                        subset_size
                    ),
                    "Validation Macro F1": (
                        score
                    ),
                    "Feature Indices": (
                        list(subset)
                    ),
                })

                print(
                    f"New best: "
                    f"F1={score:.4f}, "
                    f"features={subset}"
                )

    return (
        np.asarray(
            best_subset,
            dtype=int,
        ),
        best_score,
        evaluated_subsets,
        pd.DataFrame(history),
    )

In [14]:
BRUTE_FORCE_TOP_K = 10
BRUTE_FORCE_MINIMUM_SIZE = 3
BRUTE_FORCE_MAXIMUM_SIZE = 5

brute_prefilter = SelectKBest(
    score_func=f_classif,
    k=BRUTE_FORCE_TOP_K,
)

brute_prefilter.fit(
    X_feature_train,
    y_feature_train,
)

brute_candidate_indices = (
    brute_prefilter.get_support(
        indices=True
    )
)

print(
    "Brute-force candidate indices:",
    brute_candidate_indices,
)

start_time = time.time()

(
    brute_force_indices,
    brute_force_validation_f1,
    evaluated_subsets,
    brute_force_history_df,
) = exhaustive_feature_selection(
    candidate_indices=(
        brute_candidate_indices
    ),
    minimum_subset_size=(
        BRUTE_FORCE_MINIMUM_SIZE
    ),
    maximum_subset_size=(
        BRUTE_FORCE_MAXIMUM_SIZE
    ),
)

elapsed_time = time.time() - start_time

record_selection(
    method="Brute Force",
    family="Exhaustive",
    feature_indices=(
        brute_force_indices
    ),
    validation_macro_f1=(
        brute_force_validation_f1
    ),
    elapsed_time=elapsed_time,
)

print(
    "Total subsets evaluated:",
    evaluated_subsets,
)

brute_force_history_df

Brute-force candidate indices: [198 232 274 301 337 352 388 436 482 485]
Testing all subsets of size 3...
New best: F1=0.1393, features=(198, 232, 274)
New best: F1=0.1566, features=(198, 232, 337)
New best: F1=0.1568, features=(198, 274, 301)
New best: F1=0.1719, features=(198, 274, 337)
Testing all subsets of size 4...
New best: F1=0.1792, features=(198, 274, 337, 352)
New best: F1=0.1834, features=(232, 274, 388, 485)
Testing all subsets of size 5...
Brute Force: 4 features, validation Macro F1=0.1834, time=2.63s
Total subsets evaluated: 582


,Evaluated Subsets,Subset Size,Validation Macro F1,Feature Indices
0,1,3,0.139288,"[198, 232, 274]"
1,3,3,0.156583,"[198, 232, 337]"
2,9,3,0.156820,"[198, 274, 301]"
3,10,3,0.171925,"[198, 274, 337]"
4,155,4,0.179218,"[198, 274, 337, 352]"
5,222,4,0.183449,"[232, 274, 388, 485]"


In [15]:
selection_results_df = pd.DataFrame(
    selection_results
)

selection_results_df = (
    selection_results_df
    .sort_values(
        "Validation Macro F1",
        ascending=False,
    )
    .reset_index(drop=True)
)

selection_results_df

,Method,Family,Selected Features,Validation Macro F1,Selection Time (s)
0,L1-SVM C=0.1,L1-SVM,464,0.382922,5.451803
1,ANOVA K=416,ANOVA,416,0.379487,1.973923
2,All Features,Baseline,512,0.379398,2.692693
3,ANOVA K=480,ANOVA,480,0.374801,2.508238
4,ANOVA K=496,ANOVA,496,0.372806,2.613479
5,ANOVA K=448,ANOVA,448,0.359546,2.265706
6,ANOVA K=384,ANOVA,384,0.359125,1.640140
7,ANOVA K=320,ANOVA,320,0.358512,1.253332
8,ANOVA K=256,ANOVA,256,0.346701,0.889905
9,ANOVA K=192,ANOVA,192,0.340055,0.557983


In [16]:
best_family_indices = (
    selection_results_df
    .groupby("Family")[
        "Validation Macro F1"
    ]
    .idxmax()
)

best_family_rows = (
    selection_results_df
    .loc[best_family_indices]
    .sort_values(
        "Validation Macro F1",
        ascending=False,
    )
    .reset_index(drop=True)
)

best_family_rows

,Method,Family,Selected Features,Validation Macro F1,Selection Time (s)
0,L1-SVM C=0.1,L1-SVM,464,0.382922,5.451803
1,ANOVA K=416,ANOVA,416,0.379487,1.973923
2,All Features,Baseline,512,0.379398,2.692693
3,Sequential Forward,Sequential,15,0.237847,2.157138
4,Brute Force,Exhaustive,4,0.183449,2.629523


In [17]:
FINAL_FEATURE_SETS = {}

for _, row in best_family_rows.iterrows():
    method_name = row["Method"]

    if method_name == "All Features":
        final_indices = np.arange(
            X_train.shape[1]
        )

    elif method_name.startswith("ANOVA K="):
        k = int(
            method_name.split("=")[1]
        )

        final_selector = SelectKBest(
            score_func=f_classif,
            k=k,
        )

        final_selector.fit(
            X_train,
            y_train,
        )

        final_indices = (
            final_selector.get_support(
                indices=True
            )
        )

    elif method_name.startswith("L1-SVM C="):
        c_value = float(
            method_name.split("=")[1]
        )

        final_l1_scaler = StandardScaler()

        X_train_l1 = (
            final_l1_scaler.fit_transform(
                X_train
            )
        )

        final_l1_model = LinearSVC(
            C=c_value,
            penalty="l1",
            dual=False,
            class_weight="balanced",
            tol=1e-3,
            max_iter=20000,
            random_state=42,
        )

        final_l1_model.fit(
            X_train_l1,
            y_train,
        )

        final_mask = np.any(
            np.abs(
                final_l1_model.coef_
            ) > 1e-8,
            axis=0,
        )

        final_indices = np.flatnonzero(
            final_mask
        )

    else:
        # Sequential and brute-force subsets
        # were selected using the inner split.
        final_indices = (
            selected_feature_sets[
                method_name
            ]
        )

    FINAL_FEATURE_SETS[
        method_name
    ] = np.asarray(
        final_indices,
        dtype=int,
    )

    print(
        method_name,
        "->",
        len(final_indices),
        "features",
    )

L1-SVM C=0.1 -> 481 features
ANOVA K=416 -> 416 features
All Features -> 512 features
Sequential Forward -> 15 features
Brute Force -> 4 features


In [18]:
MODELS = {
    "MLP": make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(256, 128),
            early_stopping=True,
            max_iter=300,
            random_state=42,
        ),
    ),

    "KNN": make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(
            n_neighbors=15,
            metric="cosine",
            n_jobs=-1,
        ),
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),

    "Linear SVM": make_pipeline(
        StandardScaler(),
        LinearSVC(
            class_weight="balanced",
            max_iter=10000,
            random_state=42,
        ),
    ),
}

In [19]:
final_results = []

for method_name, feature_indices in (
    FINAL_FEATURE_SETS.items()
):
    print("\n" + "=" * 70)

    print(
        f"{method_name}: "
        f"{len(feature_indices)} features"
    )

    X_train_selected = X_train[
        :,
        feature_indices,
    ]

    X_test_selected = X_test[
        :,
        feature_indices,
    ]

    for model_name, base_model in (
        MODELS.items()
    ):
        print(
            f"Training {model_name}..."
        )

        model = clone(base_model)

        start_time = time.time()

        model.fit(
            X_train_selected,
            y_train,
        )

        predictions = model.predict(
            X_test_selected
        )

        elapsed_time = (
            time.time() - start_time
        )

        final_results.append({
            "Embedding": DATASET_NAME,
            "Feature Method": (
                method_name
            ),
            "Selected Features": len(
                feature_indices
            ),
            "Model": model_name,
            "Accuracy": accuracy_score(
                y_test,
                predictions,
            ),
            "Balanced Accuracy": (
                balanced_accuracy_score(
                    y_test,
                    predictions,
                )
            ),
            "Macro F1": f1_score(
                y_test,
                predictions,
                average="macro",
                zero_division=0,
            ),
            "Train/Eval Time (s)": (
                elapsed_time
            ),
        })


L1-SVM C=0.1: 481 features
Training MLP...
Training KNN...
Training Random Forest...
Training Linear SVM...

ANOVA K=416: 416 features
Training MLP...
Training KNN...
Training Random Forest...
Training Linear SVM...

All Features: 512 features
Training MLP...
Training KNN...
Training Random Forest...
Training Linear SVM...

Sequential Forward: 15 features
Training MLP...
Training KNN...
Training Random Forest...
Training Linear SVM...

Brute Force: 4 features
Training MLP...
Training KNN...
Training Random Forest...
Training Linear SVM...


In [20]:
final_results_df = pd.DataFrame(
    final_results
)

final_results_df = (
    final_results_df
    .sort_values(
        "Macro F1",
        ascending=False,
    )
    .reset_index(drop=True)
)

final_results_df

,Embedding,Feature Method,Selected Features,Model,Accuracy,Balanced Accuracy,Macro F1,Train/Eval Time (s)
0,Whisper Base 3.0s,All Features,512,MLP,0.492733,0.503076,0.500115,2.040639
1,Whisper Base 3.0s,ANOVA K=416,416,MLP,0.495640,0.499829,0.497390,2.725984
2,Whisper Base 3.0s,L1-SVM C=0.1,481,MLP,0.479651,0.488758,0.489856,1.735134
3,Whisper Base 3.0s,All Features,512,Linear SVM,0.437500,0.448135,0.445268,4.392852
4,Whisper Base 3.0s,ANOVA K=416,416,Linear SVM,0.437500,0.453462,0.445186,2.872313
5,Whisper Base 3.0s,L1-SVM C=0.1,481,Random Forest,0.444767,0.454619,0.444571,1.427555
6,Whisper Base 3.0s,L1-SVM C=0.1,481,Linear SVM,0.425872,0.439607,0.432445,4.057172
7,Whisper Base 3.0s,ANOVA K=416,416,Random Forest,0.420058,0.433049,0.422793,1.365561
8,Whisper Base 3.0s,All Features,512,Random Forest,0.417151,0.429238,0.420482,1.497728
9,Whisper Base 3.0s,Sequential Forward,15,Random Forest,0.363372,0.370535,0.362366,0.509370


In [21]:
best_method_indices = (
    final_results_df
    .groupby("Feature Method")[
        "Macro F1"
    ]
    .idxmax()
)

best_result_per_method = (
    final_results_df
    .loc[best_method_indices]
    .sort_values(
        "Macro F1",
        ascending=False,
    )
    .reset_index(drop=True)
)

best_result_per_method[
    [
        "Feature Method",
        "Selected Features",
        "Model",
        "Accuracy",
        "Balanced Accuracy",
        "Macro F1",
        "Train/Eval Time (s)",
    ]
]

,Feature Method,Selected Features,Model,Accuracy,Balanced Accuracy,Macro F1,Train/Eval Time (s)
0,All Features,512,MLP,0.492733,0.503076,0.500115,2.040639
1,ANOVA K=416,416,MLP,0.495640,0.499829,0.497390,2.725984
2,L1-SVM C=0.1,481,MLP,0.479651,0.488758,0.489856,1.735134
3,Sequential Forward,15,Random Forest,0.363372,0.370535,0.362366,0.509370
4,Brute Force,4,Random Forest,0.209302,0.212991,0.208639,0.443460


In [22]:
BASELINE_MACRO_F1 = 0.5001

comparison_df = (
    best_result_per_method.copy()
)

comparison_df[
    "Macro F1 Change"
] = (
    comparison_df["Macro F1"]
    - BASELINE_MACRO_F1
)

comparison_df[
    "Feature Reduction (%)"
] = (
    1
    - (
        comparison_df[
            "Selected Features"
        ]
        / 512
    )
) * 100

comparison_df[
    [
        "Feature Method",
        "Selected Features",
        "Model",
        "Macro F1",
        "Macro F1 Change",
        "Feature Reduction (%)",
    ]
]

,Feature Method,Selected Features,Model,Macro F1,Macro F1 Change,Feature Reduction (%)
0,All Features,512,MLP,0.500115,0.000015,0.000000
1,ANOVA K=416,416,MLP,0.497390,-0.002710,18.750000
2,L1-SVM C=0.1,481,MLP,0.489856,-0.010244,6.054688
3,Sequential Forward,15,Random Forest,0.362366,-0.137734,97.070312
4,Brute Force,4,Random Forest,0.208639,-0.291461,99.218750


In [23]:
OUTPUT_DIR = Path(
    "whisper_3_0s_feature_selection_results"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

selection_results_df.to_csv(
    OUTPUT_DIR
    / "validation_feature_selection.csv",
    index=False,
)

final_results_df.to_csv(
    OUTPUT_DIR
    / "heldout_test_results.csv",
    index=False,
)

best_result_per_method.to_csv(
    OUTPUT_DIR
    / "best_result_per_method.csv",
    index=False,
)

comparison_df.to_csv(
    OUTPUT_DIR
    / "baseline_comparison.csv",
    index=False,
)

sequential_history_df.to_csv(
    OUTPUT_DIR
    / "sequential_history.csv",
    index=False,
)

brute_force_history_df.to_csv(
    OUTPUT_DIR
    / "brute_force_history.csv",
    index=False,
)

np.savez(
    OUTPUT_DIR
    / "selected_feature_indices.npz",
    **{
        method_name
        .lower()
        .replace(" ", "_")
        .replace("=", "_")
        .replace(".", "_"): indices

        for method_name, indices
        in FINAL_FEATURE_SETS.items()
    },
)

print(
    "Saved results to:",
    OUTPUT_DIR.resolve(),
)

Saved results to: /Users/bhavaykhatri/Desktop/Assignments/audio_data_benchmarking_mml_lab/whisper_3_0s_feature_selection_results
